base code

In [ ]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# model_name = "google/flan-t5-small"

# # Load tokenizer + model
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# # Prompt (important!)
# prompt = """Classify sentiment as  or Negative.
# Text: I love this product.
# Answer:"""

# # Tokenize
# inputs = tokenizer(prompt, return_tensors="pt")

# # Generate output
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=5,
#     do_sample=False
# )

# # Decode result
# result = tokenizer.decode(outputs[0], skip_special_tokens=True)

# print(result)

Initialization

In [41]:
# import libraries
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings("ignore")

# set config
results_df = pd.DataFrame(columns=["prompt_template", "accuracy", "F1_score"])
model_name = "google/flan-t5-small"


In [2]:
# Load the dataset
train = pd.read_csv('data/6/train.csv')
train.drop(columns=['data_id'], inplace=True)
train.dropna(inplace=True)

# valid = pd.read_csv('data/6/valid.csv')
# valid.drop(columns=['data_id'], inplace=True)
# valid.dropna(inplace=True)

In [3]:
train_df, samples_df = train.iloc[:500].copy(), train.iloc[1000:2000].copy()
train_df.shape, samples_df.shape

((500, 2), (1000, 2))

In [ ]:
samples_df

In [ ]:
# data cleaning
def clean_text(df, text_col="text", target_col="sentiment"):

    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].str.strip()
    df[text_col] = df[text_col].str[:2048]

    df[target_col] = df[target_col].str.lower()
    return df

# inference
def batch_predict(texts, prompt_template, tokenizer, model):
    prompts = [prompt_template.format(t=t) for t in texts]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

def pipeline(df, prompt_template):
    # data cleaning
    df = clean_text(df)

    # Load model
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Set model to eval mode (faster)
    model.eval()

    # Apply in batches
    batch_size = 32
    preds = []

    for i in range(0, len(df), batch_size):
        batch_texts = df["text"].iloc[i:i+batch_size].tolist()
        preds.extend(batch_predict(batch_texts, prompt_template, tokenizer, model))

    df["predicted"] = [p.strip() for p in preds]

    # evaluation
    df['predicted'] = df['predicted'].str.lower()
    df['sentiment'] = df['sentiment'].str.lower()
    df['correct'] = df['sentiment'] == df['predicted']

    f1 = f1_score(df['sentiment'], df['predicted'], average='weighted')
    print(f"F1-Score: {round(100*f1, 1)}%")

    results = {
        "prompt_template": prompt_template,
        "F1_score": f1
    }

    return results, df

## Final Prompts

In [47]:
# one shot
# f1 = 91.5
one_shot_prompt = """You are a sentiment classifier.
Answer with only one word: positive or negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 91.5%


In [48]:
# however
# f1 = 91.6
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive , negative

Answer with exactly one word: positive or negative. Do not explain.
After "but", "however", or "although", the following sentiment is more important.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 91.6%


In [ ]:
# two shot
# f1 = 91
two_shot_prompt = """You are a sentiment classifier.
Answer with only one word: positive or negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: negative

Text: {t}
Answer:"""

results_two_shot, df = pipeline(train_df, two_shot_prompt)

F1-Score: 91.0%


In [75]:
# question
# f1 = 91
prompt = """
What is the sentiment of this text?

Answer using only one word: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 91.0%


In [ ]:
# explain sentiment
# f1 = 90.2
prompt = """
Classify sentiment.

Positive = praise, satisfaction, good experience.
Negative = complaint, disappointment, bad experience.

Answer with one word only: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 90.2%


## GPT

In [49]:
prompt = """
Classify the sentiment of the text.

Labels: positive, negative

Answer with exactly one word: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 90.1%


In [50]:
prompt = """
Determine whether the sentiment of the following text is positive or negative.

Respond with only one word: positive or negative. Do not explain.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 86.2%


In [51]:
prompt = """
Sentiment classification task.

Output format: one word only (positive or negative).

Text: {t}
Sentiment:"""

results, df = pipeline(train_df, prompt)

F1-Score: 90.3%


In [52]:
prompt = """
What is the sentiment of this text?

Answer using only one word: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 91.0%


In [53]:
prompt = """
Classify sentiment.

Allowed answers: positive, negative.
Output only one word from the allowed answers.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 89.4%


In [54]:
prompt = """
Classify the sentiment of the text.

If the text expresses satisfaction, choose positive.
If it expresses dissatisfaction, choose negative.

Answer with exactly one word: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 89.9%


In [55]:
prompt = """
Text: {t}

Is the sentiment positive or negative?
Answer with one word only.
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 87.2%


In [56]:
prompt = """
Classify the sentiment of the text.

Focus on the final opinion, especially after words like "but", "however", or "although".

Answer with exactly one word: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 90.9%


In [57]:
prompt = """
You are a strict sentiment classifier.

Your output must be exactly one word: positive or negative.

Do not explain. Do not add punctuation.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 41.2%


In [58]:
prompt = """
Classify sentiment.

Positive = praise, satisfaction, good experience.
Negative = complaint, disappointment, bad experience.

Answer with one word only: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 90.2%


In [59]:
prompt = """
Sentiment?

positive or negative only.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 90.2%


In [60]:
prompt = """
Classify the sentiment carefully.

Ignore neutral descriptions. Focus on opinion words.

Return exactly one word: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 89.4%


In [61]:
prompt = """
Binary sentiment classification.

Allowed outputs: positive, negative.
Only output one of these words.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 87.9%


In [62]:
prompt = """
Instruction: classify the sentiment of the input text.

Response format: a single word (positive or negative).

Input: {t}
Output:"""

results, df = pipeline(train_df, prompt)

F1-Score: 89.6%


In [63]:
prompt = """
Classify sentiment quickly and directly.

Do not explain or justify.

Answer with exactly one word: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 88.1%


In [65]:
prompt = """
Classify the sentiment of the text.

Be strict when detecting negativity:
- If there is any complaint, disappointment, problem, or criticism, classify as negative.
- If the text contains both positive and negative parts, the final sentiment is negative.
- Pay special attention to words like "but", "however", or "although" — the sentiment after them is the true sentiment.

Only classify as positive if the text is clearly and purely positive with no negative cues.

Answer with exactly one word: positive or negative.

Text: {t}
Answer:"""

results, df = pipeline(train_df, prompt)

F1-Score: 89.4%


In [70]:
df[df['correct'] == False & df['sentiment'] == 'negative']

TypeError: Cannot perform 'rand_' with a dtyped [object] array and scalar of type [bool]

In [72]:
len(df[(df['correct'] == False) & (df['sentiment'] == 'negative')])

10

In [73]:
prompt = """
Classify the overall sentiment of the review.

Rules:
- Answer with exactly one word: positive or negative.
- If the text includes any complaint, defect, failure, criticism, disappointment, or recommendation against the product, classify it as negative.
- If positive and negative parts both appear, focus on the final overall judgment.
- Pay special attention to contrast words such as "but", "however", "although", and "though". The sentiment after them is often more important.
- Do not be influenced by a few positive words if the review describes a problem.
- Treat implied criticism such as poor quality, broken parts, weak durability, or regret as negative.

Text: {t}
Answer:"""

results, df2 = pipeline(train_df, prompt)
print(len(df2[(df2['correct'] == False) & (df2['sentiment'] == 'negative')]))

F1-Score: 89.6%
11


In [67]:
df[df['correct'] == False][['text', 'sentiment', 'predicted']].to_csv('data/gpt_misclassified_samples.csv', index=False)

## Old codes

Baseline

In [5]:
prompt_template = f"Classify sentiment as positive or negative.\nText: {{t}}\nAnswer:"
results, df = pipeline(train_df, prompt_template)

F1-Score: 89.9%


Stronger zero-shot prompt

In [6]:
zero_shot_prompt = """You are a sentiment classifier.
Classify the sentiment of the text as positive or negative.
Answer with only one word: positive or negative.

Text: {t}
Answer:"""
results, df = pipeline(train_df, zero_shot_prompt)


F1-Score: 90.1%


Few-shot prompt

In [21]:
positive_sample = samples_df[samples_df['sentiment'] == 'positive'].sample(2, random_state=2501676)
clean_povitive_examples = clean_text(positive_sample)

negative_sample = samples_df[samples_df['sentiment'] == 'negative'].sample(1, random_state=2501676)
clean_negative_examples = clean_text(negative_sample)

print("positive Examples:")
print(clean_povitive_examples['text'].tolist()[0])
print(clean_povitive_examples['text'].tolist()[1])
print("\nnegative Example:")
print(clean_negative_examples['text'].tolist()[0])

positive Examples:
luv it
It’s subtle and looks good enough to use daily. I was really struggling with the mustiness of my front loader. This has helped tremendously. I’d buy another in a second if I lost it.

negative Example:
Dishes need to be thoroughly scrubbed prior. Dish washing times are almost twice as long as listed in the manual. Too many setting options and not much of a difference when they come out. I'm now using the lower setting with faster wash and comes out cleaner than the heavy wash cycle but saves water. Some parts are made of low quality plastic. A small peice on a &#34;pod&#34; basket broke off on day 1 but I won't use that anyway. An 11 inch dish is a very tight fit. It does steam heat when you use cold water. All in all it does save time from standing over the sink and doing dishes all day.


In [13]:
one_shot_prompt = """You are a sentiment classifier.
Answer with only one word: positive or negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 91.5%


In [64]:
two_shot_prompt = """You are a sentiment classifier.
Answer with only one word: positive or negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: negative

Text: {t}
Answer:"""

results_two_shot, df = pipeline(train_df, two_shot_prompt)

F1-Score: 91.0%


In [16]:
few_shot_prompt = """You are a sentiment classifier.
Answer with only one word: positive or negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: positive

Text: This item worked perfectly for my refrigerator.
Answer: positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: negative

Text: {t}
Answer:"""

results_few, df = pipeline(train_df, few_shot_prompt)

F1-Score: 91.0%


Label-definition prompt

In [34]:
lable_definition_prompt = """Classify the sentiment of the text.

positive = happy, satisfied, favorable
negative = unhappy, dissatisfied, unfavorable

Answer with only one word: positive or negative.

Text: {t}
Answer:"""
results_lable_definition, df = pipeline(train_df, lable_definition_prompt)

print(results_lable_definition)

F1-Score: 87.7%
{'prompt_template': 'Classify the sentiment of the text.\n\npositive = happy, satisfied, favorable\nnegative = unhappy, dissatisfied, unfavorable\n\nAnswer with only one word: positive or negative.\n\nText: {t}\nAnswer:', 'accuracy': np.float64(0.868), 'F1_score': 0.8767720505525044}


Different Examples in one shot

In [23]:
positive_samples = samples_df[samples_df['sentiment'] == 'positive'].sample(10, random_state=2501676)
clean_povitive_examples = clean_text(positive_samples)
for exp in clean_povitive_examples['text']:
    print(exp)

luv it
It’s subtle and looks good enough to use daily. I was really struggling with the mustiness of my front loader. This has helped tremendously. I’d buy another in a second if I lost it.
Good to use it
Easier to install than I anticipated.
Exactly what I needed plus a few extra maintenance parts to make sure the machine would last at an excellent price.
Fit
Capsules made of god quality and i can put any kind of coffee
Perfect fit.
fits great in the dryer! great product!
Been using the Melitta Java Jig and Filters for a number of years, I will recommend them to anyone that likes to use their coffee and not the prepackage K-Cup.


Check different examples

In [43]:
positive_samples = samples_df[samples_df['sentiment'] == 'positive'].sample(10, random_state=2501676)
clean_povitive_examples = clean_text(positive_samples)
f1_scores = []

for exp in clean_povitive_examples['text']:
    one_shot_prompt = f"""You are a sentiment classifier.
    Answer with only one word: positive or negative.

    Text: {exp}
    Answer: positive

    """+"""Text: {t}
    Answer:"""
    results_few, df = pipeline(train_df, one_shot_prompt)
    f1_scores.append(results_few['F1_score'])    

print(np.mean(f1_scores))

F1-Score: 90.2%
F1-Score: 90.7%
F1-Score: 90.6%
F1-Score: 90.8%
F1-Score: 91.3%
F1-Score: 91.0%
F1-Score: 90.0%
F1-Score: 91.0%
F1-Score: 91.1%
F1-Score: 91.3%
0.9080029022427087


Prompt paraphrasing

In [ ]:
samples_df[samples_df['sentiment'] == 'negative']['text'].tolist()[:5]

Baseline on 4000 samples is 0.9

In [45]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive , negative

Answer with exactly one word: positive or negative. Do not explain.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 90.9%


Baseline on 1000 samples is 0.9

Error Analysis using LLM

In [ ]:
df[df['correct'] == False][['text', 'sentiment', 'predicted']].to_csv('data/gpt_misclassified.csv', index=False)

In [ ]:
misclassified_df = df[df['correct'] == False][['text', 'sentiment', 'predicted']]
misclassified_df['sentiment'].value_counts()

In [ ]:
misclassified_df[misclassified_df['sentiment'] == '']['text']

Add negative example failed -> 0.87

In [ ]:
one_shot_prompt = """Classify the sentiment of the following text as either  or negative.

If the text contains both  and negative statements, choose the label that best matches the overall sentiment.  Return exactly one lowercase word (“” or “negative”).  Do not explain your answer.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Sentiment: 

Example:
Text: This thing works pretty good but it's extremely loud. I have it in my office at work and sometimes it get annoying its so loud. It's pretty cheap so I would recommend putting it in a shop but not in an office.
Sentiment: negative

Now classify:
Text: {t}
Sentiment:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

In [ ]:
one_shot_prompt = """Classify the sentiment of the following text as either  or negative.

If the text contains both  and negative statements, choose the label that best matches the overall sentiment.  Return exactly one lowercase word (“” or “negative”).  Do not explain your answer.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Sentiment: 

Now classify:
Text: {t}
Sentiment:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

Add guide about handling but, however, etc -> Improved: 0.91

In [46]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive , negative

Answer with exactly one word: positive or negative. Do not explain.
After "but", "however", or "although", the following sentiment is more important.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 91.6%


In [ ]:
df[df['correct'] == False][['text', 'sentiment', 'predicted']].to_csv('data/gpt_misclassified.csv', index=False)

In [ ]:
Classify the sentiment of the text.

If the text is unclear, very short, or lacks strong sentiment words, label it as negative.

Labels: , negative  
Answer with exactly one word.

Text: {t}
Answer:

## Not in thread

Add: If you were unsure -> 

In [ ]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: , negative

Answer with exactly one word:  or negative. Do not explain.
After "but", "however", or "although", the following sentiment is more important.
If the text contains both  and negative statements, or if the sentiment is unclear, choose .
Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: 

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

In [ ]:
df[df['correct'] == False][['text', 'sentiment', 'predicted']].to_csv('data/gpt_misclassified.csv', index=False)

Oposite sentiment of phrase before 'but'

In [ ]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: , negative

Answer with exactly one word:  or negative. Do not explain.
If the text contains "but", "however", or "although", the final sentiment is usually the opposite of the sentiment before these words (e.g., a  phrase followed by "but" should be classified as negative).

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: 

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

paraphrase of the above prompt

In [ ]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: , negative

Answer with exactly one word:  or negative. Do not explain.
After "but", "however", or "although", the sentiment typically reverses from the earlier part.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: 

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

In [ ]:
df[
    (df['predicted'] == 'negative') &
    (df['sentiment'] == '') &
    (df['text'].str.contains(r'\b(but|however|although)\b', case=False, na=False))
]['text'].tolist()

In [ ]:
train_df['text'].str.lower()

In [ ]:
lower_df = train_df.copy() 
lower_df['text'] = lower_df['text'].str.lower()
lower_df

text to lower case

In [ ]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: , negative

Answer with exactly one word:  or negative. Do not explain.
After "but", "however", or "although", the sentiment typically reverses from the earlier part.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: 

Text: {t}
Answer:"""

results_one_shot, df = pipeline(lower_df, one_shot_prompt)

Dont mention it is sentiment analysis

In [ ]:
one_shot_prompt = """Classify this text into  or negative.

Answer with exactly one word:  or negative. Do not explain.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: 

Text: {t}
Answer:"""

results_one_shot, df = pipeline(lower_df, one_shot_prompt)

Add role

In [ ]:
one_shot_prompt = """
You are a sentiment classifier.
Classify this text into  or negative.

Answer with exactly one word:  or negative. Do not explain.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: 

Text: {t}
Answer:"""

results_one_shot, df = pipeline(lower_df, one_shot_prompt)

In [ ]:
train_subset = train_df.head(10)

## LLM Selection

In [ ]:
import time
import pandas as pd
from sklearn.metrics import f1_score
from transformers import pipeline

MODELS = [
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "microsoft/phi-2",
    "distilgpt2",
    "google/flan-t5-small"
]

def extract_label(s):
    s = s.lower()
    if "" in s:
        return ""
    if "negative" in s:
        return "negative"
    return "negative"

results = []

for model_name in MODELS:
    start = time.time()
    pipe = pipeline("text-generation", model=model_name, device_map="auto")

    preds = []
    for text in train_subset["text"].astype(str):
        prompt = f'Classify sentiment. Return ONLY one word:  or negative.\nText: "{text}"\nAnswer:'
        out = pipe(
            prompt,
            max_new_tokens=3,
            do_sample=False,
            return_full_text=False,
            pad_token_id=pipe.tokenizer.eos_token_id,
        )[0]["generated_text"]
        preds.append(extract_label(out))

    elapsed = round(time.time() - start, 2)
    f1 = round(f1_score(train_subset["sentiment"].str.lower(), preds, pos_label=""), 4)

    results.append([model_name, elapsed, f1])

results_df = pd.DataFrame(results, columns=["model name", "execution time (in second)", "F1-score"])
print(results_df)

In [ ]:
import time
import pandas as pd
from sklearn.metrics import f1_score
from transformers import pipeline

MODELS = [
"google/flan-t5-small"
]

def extract_label(s):
    s = s.lower()
    if "" in s:
        return ""
    if "negative" in s:
        return "negative"
    return "negative"

results = []

for model_name in MODELS:
    start = time.time()
    pipe = pipeline("text-generation", model=model_name, device_map="auto")

    preds = []
    for text in train_subset["text"].astype(str):
        prompt = f'Classify sentiment. Return ONLY one word:  or negative.\nText: "{text}"\nAnswer:'
        out = pipe(
            prompt,
            max_new_tokens=3,
            do_sample=False,
            return_full_text=False,
            pad_token_id=pipe.tokenizer.eos_token_id,
        )[0]["generated_text"]
        preds.append(extract_label(out))

    elapsed = round(time.time() - start, 2)
    f1 = round(f1_score(train_subset["sentiment"].str.lower(), preds, pos_label=""), 4)

    results.append([model_name, elapsed, f1])

results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)

print(results_df)

In [ ]:
results_df

In [ ]:
valid_df = train_df.head(10).copy()

In [ ]:
import time
import pandas as pd
from sklearn.metrics import f1_score
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

text_col = "text"
label_col = "sentiment"

MODELS = [
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "microsoft/phi-2",
    "distilgpt2",
    "google/flan-t5-small"
]

def extract(x):
    x = x.lower()
    return "" if "" in x else "negative"

results = []

for m in MODELS:
    start = time.time()
    preds = []

    if m == "Qwen/Qwen2.5-0.5B-Instruct" or m == "TinyLlama/TinyLlama-1.1B-Chat-v1.0":
        tok = AutoTokenizer.from_pretrained(m)
        model = AutoModelForCausalLM.from_pretrained(m, device_map="auto", torch_dtype="auto")

        for t in valid_df[text_col].astype(str):
            msgs = [
                {"role": "system", "content": "Answer only:  or negative."},
                {"role": "user", "content": t}
            ]
            p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = tok(p, return_tensors="pt").to(model.device)
            out = model.generate(**inp, max_new_tokens=2, do_sample=False)
            txt = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
            preds.append(extract(txt))

    elif m == "google/flan-t5-small":
        pipe = pipeline("text2text-generation", model=m, device_map="auto")
        for t in valid_df[text_col].astype(str):
            out = pipe(f" or negative: {t}", max_new_tokens=2)[0]["generated_text"]
            preds.append(extract(out))

    else:
        pipe = pipeline("text-generation", model=m, device_map="auto")
        for t in valid_df[text_col].astype(str):
            out = pipe(
                f'Classify sentiment. Return ONLY one word:  or negative.\nText: "{t}"\nAnswer:',
                max_new_tokens=2,
                do_sample=False,
                return_full_text=False,
                pad_token_id=pipe.tokenizer.eos_token_id
            )[0]["generated_text"]
            preds.append(extract(out))

    f1 = f1_score(valid_df[label_col].str.lower(), preds, average="macro")
    elapsed = round(time.time() - start, 2)
    results.append([m, elapsed, round(f1, 4)])

print(pd.DataFrame(results, columns=["model", "time_sec", "F1"]))

In [ ]:
# TinyLlama/TinyLlama-1.1B-Chat-v1.0
import time
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")

def extract(x):
    x = x.lower()
    return "" if "" in x else "negative"

start = time.time()
preds = []

for t in valid_df["text"].astype(str):
    msgs = [
        {"role": "system", "content": "You are a sentiment classifier."},
        {"role": "assistant", "content": "Answer ONLY one word:  or negative."},
        {"role": "user", "content": t}
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=2, do_sample=False)
    txt = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    preds.append(extract(txt))

f1 = f1_score(valid_df["sentiment"].str.lower(), preds, average="macro")
print("F1:", round(f1,4), "Time:", round(time.time()-start,2))

In [ ]:
    msgs = [
        {"role": "system", "content": "You are a sentiment classifier."},
        {"role": "user", "Classify the sentiment of content. Answer ONLY one word:  or negative. content": t}
    ]

In [ ]:
# distilgpt2
import time
from sklearn.metrics import f1_score
from transformers import pipeline

pipe = pipeline("text-generation", model="distilgpt2")

def extract(x):
    x = x.lower()
    return "" if "" in x else "negative"

start = time.time()
preds = []

for t in valid_df["text"].astype(str):
    prompt = f'Review sentiment classification.\nText: "{t}"\nLabel (/negative):'
    out = pipe(prompt, max_new_tokens=3, do_sample=False, return_full_text=False,
               pad_token_id=pipe.tokenizer.eos_token_id)[0]["generated_text"]
    preds.append(extract(out))

f1 = f1_score(valid_df["sentiment"].str.lower(), preds, average="macro")
print("F1:", round(f1,4), "Time:", round(time.time()-start,2))

In [ ]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import f1_score

model_name = "distilgpt2"
device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

def pred(text):
    prompt = f"Review: {text}\nSentiment ( or negative):"
    inputs = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).lower()
    return "" if "" in gen else "negative" if "negative" in gen else "unknown"

start = time.time()

y_true = valid_df["sentiment"].str.lower().tolist()
y_pred = [pred(t) for t in valid_df["text"]]

print("F1:", f1_score(y_true, y_pred, average="macro"))
print("Time:", round(time.time() - start, 2), "sec")

In [ ]:
valid_df.to_csv('data/samples.csv', index=False)

In [ ]:
import pandas as pd, time
from transformers import pipeline
from sklearn.metrics import f1_score


# Models and their pipeline tasks
models = [
    ("Qwen/Qwen2.5-0.5B-Instruct", "text-generation"),
    ("google/flan-t5-small", "text2text-generation"),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "text-generation"),
    ("distilgpt2", "text-generation"),
    ("microsoft/phi-2", "text-generation"),
]

results = []

for model_name, task in models:
    start_time = time.time()
    try:
        pipe = pipeline(task, model=model_name, tokenizer=model_name)
        predictions = []
        for text in valid_df["text"]:
            prompt = (
                "Classify the sentiment of the following text as either  or negative.\n"
                f"Text: {text}\n"
                "Sentiment:"
            )
            output = pipe(prompt, max_new_tokens=2, return_full_text=False)[0]["generated_text"].lower()
            label = None
            for option in ("", "negative"):
                if option in output:
                    label = option
                    break
            predictions.append(label or "negative")
        f1 = f1_score(valid_df["sentiment"], predictions, pos_label="", average="binary")
        elapsed = time.time() - start_time
        if elapsed <= 60:
            results.append({"model": model_name, "f1_score": f1, "time_sec": elapsed})
    except Exception:
        pass

print(pd.DataFrame(results))